# Сегментация посетителей фитнес-клуба

**Цель:** выявить поведенческие сегменты клиентов 
для персонализации маркетинга и снижения оттока.

**Данные:** Gym Members Exercise Dataset, 973 записи, 15 признаков.

## Содержание
- [1. Загрузка данных](#1-загрузка-данных)
- [2. EDA](#2-eda)
- [3. Preprocessing](#3-preprocessing)
- [4. Кластеризация](#4-кластеризация)
- [4.1 Кластеризация c учетом типа тренировки](#41-кластеризация-c-учетом-типа-тренировки)
- [4.2 Кластеризация без учета типа тренировки](#42-кластеризация-без-учета-типа-тренировки)
- [5. Выводы](#5-выводы)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Загрузка данных

In [ ]:
df = pd.read_csv('../data/gym_members_exercise_tracking.csv')
df.head()

## 2. EDA

In [ ]:
df.info()

In [ ]:
df.describe()

Проверка пропусков

In [ ]:
df.isna().sum()

In [ ]:
df.hist(figsize = (14, 10), bins = 20)
plt.tight_layout()
plt.show()

In [ ]:
print(df['Gender'].value_counts())
print(df['Workout_Type'].value_counts())
print(df['Experience_Level'].value_counts())

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
corr_matrix = df.corr(numeric_only=True)
cols = corr_matrix.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        if abs(corr_matrix.iloc[i, j]) >= 0.8 :
            print(f"{cols[i]} — {cols[j]}: {corr_matrix.iloc[i, j]:.2f}")

Выявлена высокая корреляция между рядом признаков, что свидетельствует
о наличии избыточной информации в данных

## 3. Preprocessing

Между Experience_Level и тренировочной активностью клиентов обнаружена сильная связь. Чтобы кластеризация не дублировала уже имеющееся разделение клиентов на группы, признак Experience_Level необходимо на время удалить. После кластеризации можно будет сравнить разделение моделью и реальными данными.

In [ ]:
experience_lvl = df['Experience_Level']

Для уменьшения размерности признакового пространства принято решение удалить столбцы:
- Weight (kg);
- Height (m);
- Session_Duration;
- Workout_Frequency;
- BMI.

Удаление признаков Weight (kg), Height (m), BMI обусловлено тем, что они не описывают поведенческую активность клиентов.

In [ ]:
corr_columns = ['Weight (kg)',
                'Height (m)',
                'Workout_Frequency (days/week)', 
                'Session_Duration (hours)',
                'Experience_Level',
                'BMI']

In [ ]:
df = df.drop(corr_columns, axis=1)

In [ ]:
df['Gender'] = df['Gender'].map({
    'Male': 0,
    'Female': 1
})

df = pd.get_dummies(df, columns=['Workout_Type'], prefix='Workout', dtype=int)

In [ ]:
df.head()

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

df_scaled = pd.DataFrame(df_scaled, columns=df.columns)

## 4. Кластеризация

### 4.1 Кластеризация c учетом типа тренировки

**Метод силуэта**

In [ ]:
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    claster_labels = kmeans.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, claster_labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, marker='o', linestyle='-', color='b')
plt.title('Определение оптимального числа кластеров методом силуэта')
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Средний коэффициент силуэта')
plt.grid(True)
plt.show()

# 4. Выводим лучший результат
best_k = k_range[np.argmax(silhouette_scores)]
print(f"Оптимальное число кластеров: {best_k} со значением коэффициента силуэта {np.max(silhouette_scores):.2f}")

**Метод локтя**

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Inertia')
plt.show()

Оба метода для опредения оптимального числа кластеров показали, что данные не имеют четкой кластерной структуры. Коэффициент силуэта равен 0.23 при k = 5, что достаточно низко. График для метода локтя убывает плавно, нет резкого перегиба. Можно заметить небольшое замедление около k = 4-5. 

Это достаточно типично для реальных данных о людях.

Для дальнейшего анализа выбрано k = 4.

Применение метода `KMeans`

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
labels = kmeans.fit_predict(df_scaled)

df['labels'] = labels
print(df['labels'].value_counts(),'\n', experience_lvl.value_counts())

Распределение по полученным кластерам равномерное. Также количество людей в 0 кластере и с опытом 3 почти равно, возможно это одни и те же клиенты.

In [ ]:
print(pd.crosstab(df['labels'], experience_lvl))

В кластере 0 действительно 70% клиентов продвинутые. В 3 кластере 147 новичков и 153 средних, продвинутых почти нет. Кластеры 1 и 2 очень похожи между собой, в основном новички и средние. 

Алгоритм выделил продвинутых в отдельный кластер, новичков и средних объединил в три разные группы, возможно там есть отличия, не объясняющиеся уровнем опыта.

In [ ]:
df.groupby('labels').mean().round(2)

### 4.2 Кластеризация без учета типа тренировки

In [ ]:
df_scaled_no_workout = df_scaled.drop(['Workout_Cardio',	'Workout_HIIT',	'Workout_Strength', 'Workout_Yoga'], axis=1)

**Метод силуэта**

In [ ]:
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    claster_labels = kmeans.fit_predict(df_scaled_no_workout)
    score = silhouette_score(df_scaled_no_workout, claster_labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_scores, marker='o', linestyle='-', color='b')
plt.title('Определение оптимального числа кластеров методом силуэта')
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Средний коэффициент силуэта')
plt.grid(True)
plt.show()

# 4. Выводим лучший результат
best_k = k_range[np.argmax(silhouette_scores)]
print(f"Оптимальное число кластеров: {best_k} со значением коэффициента силуэта {np.max(silhouette_scores):.2f}")

**Метод локтя**

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled_no_workout)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Inertia')
plt.show()

Результаты этого эксперимента: силуэт 0.2 при двух кластерах, линейно убывающий график в методе локтя, говорят о том, что без учета типа тренировок данные абсолютно не делятся на кластеры. Значит: тип тренировок - основное разделяющий признак в этих данных.

## 5. Визуализация

In [ ]:
pca = PCA(n_components=2, random_state=42)
df_pca = pca.fit_transform(df_scaled)

plt.figure(figsize=(14,10))
colors = ['red','blue', 'green', 'pink']
for i in range(4):
    mask = df['labels'] == i
    plt.scatter(df_pca[mask, 0], df_pca[mask, 1], 
                c=colors[i], label=f'Кластер {i}', alpha=1)
plt.legend()
plt.title('Визуализация кластеров (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print(f"PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"PC2: {pca.explained_variance_ratio_[1]:.1%}")

Первые две главные компоненты объясняют лишь 31.8% дисперсии, поэтому визуализация даёт неполную картину. Розовый (3) и зеленый (2) кластеры  хорошо отделяется даже в 2D, тогда как кластеры 0 и 1 разделены в пространстве большей размерности.